# Zero-Shot High-Resolution Inference (0.25° ERA5)

Trained on 1.5° (120×240), inference on 0.25° (720×1440).

- **FNO**: Resolution-invariant by design (Fourier spectral layers)
- **AFNO**: Requires positional embedding interpolation (15×30 → 90×180 patches)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from neuralop.models import FNO
from physicsnemo.models.afno import AFNO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load trained checkpoints

In [ ]:
# Load checkpoints (trained on 1.5deg, 120x240)
fno_ckpt = torch.load("checkpoints/fno_baseline.pt", map_location="cpu", weights_only=False)
afno_ckpt = torch.load("checkpoints/afno_baseline.pt", map_location="cpu", weights_only=False)

cfg = afno_ckpt["config"]
mean = afno_ckpt["normalization_mean"]  # (1, C, 1, 1)
std = afno_ckpt["normalization_std"]    # (1, C, 1, 1)

n_vars = mean.shape[1]
print(f"Number of variables: {n_vars}")
print(f"Normalization mean: {mean.squeeze()}")
print(f"Normalization std:  {std.squeeze()}")

In [ ]:
# Rebuild FNO model and load weights
fno_model = FNO(
    n_modes=(cfg["fno_modes"], cfg["fno_modes"]),
    in_channels=n_vars,
    out_channels=n_vars,
    hidden_channels=cfg["fno_width"],
    n_layers=cfg["fno_layers"],
).to(device)
fno_model.load_state_dict(fno_ckpt["model_state_dict"])
fno_model.eval()
print(f"FNO loaded: {sum(p.numel() for p in fno_model.parameters()):,} params")

In [ ]:
# Rebuild AFNO with a wrapper that interpolates positional embeddings
# for arbitrary input resolutions

class AFNOHighRes(torch.nn.Module):
    """Wrapper around AFNO that interpolates positional embeddings for new resolutions."""

    def __init__(self, afno_model, train_h, train_w, patch_size):
        super().__init__()
        self.afno = afno_model
        self.train_h = train_h
        self.train_w = train_w
        self.patch_size = patch_size
        # Grid of patches at training resolution
        self.train_ph = train_h // patch_size
        self.train_pw = train_w // patch_size

    def forward(self, x):
        B, C, H, W = x.shape
        new_ph = H // self.patch_size
        new_pw = W // self.patch_size

        # Patch embedding (Conv2d works on any spatial size)
        x_embed = self.afno.patch_embed.proj(x)  # (B, embed_dim, new_ph, new_pw)
        x_embed = x_embed.flatten(2).transpose(1, 2)  # (B, new_ph*new_pw, embed_dim)

        # Interpolate positional embeddings from training grid to new grid
        pos = self.afno.pos_embed  # (1, train_ph*train_pw, embed_dim)
        embed_dim = pos.shape[-1]
        pos_2d = pos.reshape(1, self.train_ph, self.train_pw, embed_dim).permute(0, 3, 1, 2)
        pos_2d = F.interpolate(pos_2d, size=(new_ph, new_pw), mode="bicubic", align_corners=False)
        pos_interp = pos_2d.permute(0, 2, 3, 1).reshape(1, new_ph * new_pw, embed_dim)

        x_embed = x_embed + pos_interp

        # Run through AFNO blocks
        for blk in self.afno.blocks:
            x_embed = blk(x_embed)

        # Head: project back to pixel space
        x_embed = self.afno.head(x_embed)  # (B, new_ph*new_pw, patch_size**2 * out_channels)

        # Reshape to image
        out_channels = self.afno.head.out_features // (self.patch_size ** 2)
        x_out = x_embed.reshape(B, new_ph, new_pw, self.patch_size, self.patch_size, out_channels)
        x_out = x_out.permute(0, 5, 1, 3, 2, 4).reshape(B, out_channels, H, W)
        return x_out


# Build base AFNO at training resolution, load weights, then wrap
afno_base = AFNO(
    inp_shape=[120, 240],
    in_channels=n_vars,
    out_channels=n_vars,
    patch_size=[cfg["afno_patch_size"], cfg["afno_patch_size"]],
    embed_dim=cfg["afno_embed_dim"],
    depth=cfg["afno_depth"],
    num_blocks=cfg["afno_num_blocks"],
)
afno_base.load_state_dict(afno_ckpt["model_state_dict"])

afno_model = AFNOHighRes(
    afno_base, train_h=120, train_w=240, patch_size=cfg["afno_patch_size"]
).to(device).eval()

print(f"AFNO loaded with high-res wrapper")

## 2. Load 0.25° ERA5 data

Streaming a single sample from the WB2 high-res Zarr store. Grid is 721×1440, cropped to 720×1440 for patch divisibility.

In [ ]:
ERA5_HIGHRES = "gs://weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr"

print("Opening 0.25deg ERA5 Zarr store...")
ds_hr = xr.open_zarr(ERA5_HIGHRES, chunks={"time": 1}, storage_options=dict(token="anon"))
print(f"Variables: {list(ds_hr.data_vars)}")
print(f"Dims: {dict(ds_hr.dims)}")

# Check coordinate names
print(f"Coordinates: {list(ds_hr.coords)}")

In [ ]:
# Fetch a single input/target pair at 24h lead time from validation period
# Using 2017-01-15 00:00 as input, 2017-01-16 00:00 as target
lead_hours = 24
input_time = "2017-01-15T00:00"
target_time = "2017-01-16T00:00"

variables = [
    ("geopotential", 500),
    ("temperature", 850),
    ("u_component_of_wind", 850),
    ("v_component_of_wind", 850),
]
var_names = ["z500", "t850", "u850", "v850"]

channels_in, channels_tgt = [], []
for var_name, level in variables:
    da = ds_hr[var_name].sel(level=level)
    
    print(f"Fetching {var_name}@{level}hPa for input ({input_time})...")
    inp = da.sel(time=input_time).compute().values.astype(np.float32)
    
    print(f"Fetching {var_name}@{level}hPa for target ({target_time})...")
    tgt = da.sel(time=target_time).compute().values.astype(np.float32)
    
    # Crop from 721 to 720 for patch-size divisibility
    channels_in.append(inp[:720, :])
    channels_tgt.append(tgt[:720, :])

lat_hr = ds_hr.latitude.values[:720]
lon_hr = ds_hr.longitude.values

x_hr = np.stack(channels_in, axis=0)[np.newaxis]   # (1, 4, 720, 1440)
y_hr = np.stack(channels_tgt, axis=0)[np.newaxis]   # (1, 4, 720, 1440)
print(f"\nInput shape:  {x_hr.shape}")
print(f"Target shape: {y_hr.shape}")
print(f"Grid: {len(lat_hr)} lat x {len(lon_hr)} lon")

## 3. Run inference

Normalize with the same mean/std from training (1.5°), run both models, denormalize back.

In [ ]:
x_tensor = torch.from_numpy(x_hr)
y_tensor = torch.from_numpy(y_hr)

# Normalize using training statistics
x_norm = (x_tensor - mean) / std

with torch.no_grad(), torch.amp.autocast("cuda"):
    # FNO inference (resolution-invariant)
    print("Running FNO on 720x1440...")
    fno_pred_norm = fno_model(x_norm.to(device)).cpu()
    fno_pred = fno_pred_norm * std + mean
    print(f"  FNO output shape: {fno_pred.shape}")

    # AFNO inference (with pos-embed interpolation)
    print("Running AFNO on 720x1440...")
    afno_pred_norm = afno_model(x_norm.to(device)).cpu()
    afno_pred = afno_pred_norm * std + mean
    print(f"  AFNO output shape: {afno_pred.shape}")

print("Done!")

## 4. Compute metrics (latitude-weighted RMSE)

In [ ]:
# Latitude-weighted RMSE per variable
lat_weights = np.cos(np.deg2rad(lat_hr)).astype(np.float32)
lat_weights = lat_weights / lat_weights.mean()
w = torch.from_numpy(lat_weights).view(1, 1, -1, 1)

var_units = {"z500": "m\u00b2/s\u00b2", "t850": "K", "u850": "m/s", "v850": "m/s"}

print(f"{'Model':<8} | {'Variable':<8} | {'RMSE':>10} | Unit")
print("-" * 45)
for name, pred in [("FNO", fno_pred), ("AFNO", afno_pred)]:
    se = (pred - y_tensor).pow(2)
    rmse_per_var = (se * w).mean(dim=(0, 2, 3)).sqrt()
    for i, vn in enumerate(var_names):
        print(f"{name:<8} | {vn.upper():<8} | {rmse_per_var[i].item():>10.2f} | {var_units[vn]}")

## 5. Visualize: Truth vs Prediction vs Error (0.25°)

In [ ]:
def plot_highres_comparison(pred, truth, lat, lon, model_name, var_names, var_units):
    """Plot truth / prediction / error for each variable at high resolution."""
    pred_np = pred[0].numpy()
    truth_np = truth[0].numpy()
    error_np = pred_np - truth_np
    n_vars = len(var_names)

    fig, axes = plt.subplots(n_vars, 3, figsize=(24, 7 * n_vars))
    if n_vars == 1:
        axes = axes[np.newaxis, :]

    for row, vn in enumerate(var_names):
        unit = var_units.get(vn, "")
        t, p, e = truth_np[row], pred_np[row], error_np[row]

        vmin, vmax = min(t.min(), p.min()), max(t.max(), p.max())
        eabs = max(abs(e.min()), abs(e.max()))

        im0 = axes[row, 0].pcolormesh(lon, lat, t, vmin=vmin, vmax=vmax, cmap="viridis", shading="auto")
        axes[row, 0].set_title(f"Truth - {vn.upper()} [{unit}]", fontsize=13)
        fig.colorbar(im0, ax=axes[row, 0], shrink=0.7)

        im1 = axes[row, 1].pcolormesh(lon, lat, p, vmin=vmin, vmax=vmax, cmap="viridis", shading="auto")
        axes[row, 1].set_title(f"Prediction - {vn.upper()} [{unit}]", fontsize=13)
        fig.colorbar(im1, ax=axes[row, 1], shrink=0.7)

        im2 = axes[row, 2].pcolormesh(lon, lat, e, vmin=-eabs, vmax=eabs, cmap="RdBu_r", shading="auto")
        axes[row, 2].set_title(f"Error - {vn.upper()} [{unit}]", fontsize=13)
        fig.colorbar(im2, ax=axes[row, 2], shrink=0.7)

        for ax in axes[row]:
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")

    fig.suptitle(
        f"{model_name} - 0.25\u00b0 inference ({lead_hours}h forecast, trained on 1.5\u00b0)",
        fontsize=15, y=1.01,
    )
    fig.tight_layout()
    return fig

In [ ]:
fig_fno = plot_highres_comparison(fno_pred, y_tensor, lat_hr, lon_hr, "FNO", var_names, var_units)
fig_fno.savefig("results/fno_highres_24h.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/fno_highres_24h.png")

In [ ]:
fig_afno = plot_highres_comparison(afno_pred, y_tensor, lat_hr, lon_hr, "AFNO", var_names, var_units)
fig_afno.savefig("results/afno_highres_24h.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/afno_highres_24h.png")

## 6. Side-by-side comparison (FNO vs AFNO error)

In [ ]:
# Side-by-side error comparison
fno_err = (fno_pred - y_tensor)[0].numpy()
afno_err = (afno_pred - y_tensor)[0].numpy()

fig, axes = plt.subplots(len(var_names), 2, figsize=(20, 6 * len(var_names)))
if len(var_names) == 1:
    axes = axes[np.newaxis, :]

for row, vn in enumerate(var_names):
    unit = var_units[vn]
    emax = max(abs(fno_err[row]).max(), abs(afno_err[row]).max())

    im0 = axes[row, 0].pcolormesh(lon_hr, lat_hr, fno_err[row], vmin=-emax, vmax=emax, cmap="RdBu_r", shading="auto")
    axes[row, 0].set_title(f"FNO Error - {vn.upper()} [{unit}]", fontsize=13)
    fig.colorbar(im0, ax=axes[row, 0], shrink=0.7)

    im1 = axes[row, 1].pcolormesh(lon_hr, lat_hr, afno_err[row], vmin=-emax, vmax=emax, cmap="RdBu_r", shading="auto")
    axes[row, 1].set_title(f"AFNO Error - {vn.upper()} [{unit}]", fontsize=13)
    fig.colorbar(im1, ax=axes[row, 1], shrink=0.7)

    for ax in axes[row]:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

fig.suptitle(f"Error comparison at 0.25\u00b0 ({lead_hours}h forecast, trained on 1.5\u00b0)", fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig("results/highres_error_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/highres_error_comparison.png")